<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the one third-party package this notebook needs. The thetadata client wraps the connection to ThetaData's OPRA feed, the consolidated options quote feed, and gives us the quote objects we work with.

In [ ]:
!pip install thetadata

The pip install only gets us the Python client. ThetaClient launches ThetaTerminal.jar when we construct it, and the terminal prints its connection status to the console, so the machine needs Java 11 or later plus a ThetaData account with a Standard or PRO subscription for streaming. Without Java the import still succeeds and the terminal never starts, so connect_stream has nothing to attach to.

## Imports and setup

req_quote_stream_opt takes the expiration as a date object, which is why datetime appears here. Quote stores a bid and ask with a millisecond timestamp, ThetaClient owns the connection and the subscriptions, and OptionRight.CALL or OptionRight.PUT selects the side of the strike. StreamMsgType labels each inbound message, StreamResponseType reports whether a subscription was accepted, and StreamMsg goes unused.

In [ ]:
from datetime import date

In [ ]:
import thetadata.client
from thetadata import (
    Quote,
    StreamMsg,
    ThetaClient,
    OptionRight,
    StreamMsgType,
)
from thetadata import StreamResponseType as ResponseType

## Store the latest leg quotes

Cache the most recent quote for each leg and the last rounded straddle midpoint we printed.

In [ ]:
last_call_quote = Quote()
last_put_quote = Quote()
price = 0

Each quote message from req_quote_stream_opt carries one contract, so a call update tells us nothing about the put. To price both legs we keep the last quote seen for each, and the two can be seconds apart, which this code never checks before it adds them. Both caches start as empty Quote objects with zero prices, so early prints can mix a real quote with a default one.

## Price the straddle on each update

The callback fires on every streamed message and returns at once for anything that isn't a QUOTE. It copies the incoming quote into the call or put cache, sums the two bids and the two asks, and prints only when the rounded midpoint changes, so a bid move that leaves the midpoint alone prints nothing.

In [ ]:
def callback_straddle(msg):
    if msg.type != StreamMsgType.QUOTE:
        return

    global price

    if msg.contract.isCall:
        last_call_quote.copy_from(msg.quote)
    else:
        last_put_quote.copy_from(msg.quote)

    straddle_bid = round(
        last_call_quote.bid_price + last_put_quote.bid_price, 2
    )
    straddle_ask = round(
        last_call_quote.ask_price + last_put_quote.ask_price, 2
    )
    straddle_mid = round((straddle_bid + straddle_ask) / 2, 2)

    time_stamp = thetadata.client.ms_to_time(msg.quote.ms_of_day)

    if price != straddle_mid:
        print(
            f"time: {time_stamp} bid: {straddle_bid} "
            f"mid: {straddle_mid} ask: {straddle_ask}"
        )
        price = straddle_mid

straddle_mid is an indicative mark for our own records, and nobody has to fill a two-leg order at the sum of two separately quoted midpoints. The gap between straddle_bid and straddle_ask is the number worth watching, since 15-minute delayed data drops every quote change in between and says nothing about the current spread. The comparison against the last printed midpoint is a display choice, because consecutive SPY messages often repeat the same rounded price.

## Connect and subscribe to live quotes

Open the connection, register the callback, and subscribe to both legs. The SPY 410 strike expiring May 5, 2023 has long since expired, so swap in a listed expiration or nothing ever arrives.

In [ ]:
def streaming_straddle():
    client = ThetaClient(username="", passwd="")

    client.connect_stream(callback_straddle)

    req_id_call = client.req_quote_stream_opt(
        "SPY", date(2023, 5, 5), 410, OptionRight.CALL
    )

    req_id_put = client.req_quote_stream_opt(
        "SPY", date(2023, 5, 5), 410, OptionRight.PUT
    )

    if (
        client.verify(req_id_call) != ResponseType.SUBSCRIBED
        or client.verify(req_id_put) != ResponseType.SUBSCRIBED
    ):
        raise Exception(
            "Unable to stream contract. "
            "A standard/PRO subscription required."
        )

Fill in username and passwd with our ThetaData credentials before running this. client.verify returns a response type for each request, and anything other than SUBSCRIBED means the stream was never established, which without the check looks the same as a quiet market. The exception only says the request was refused, so to tell an entitlement problem from a bad contract we read the ThetaTerminal console.

Start the stream. The callback prints only when the rounded midpoint moves, and there is no reconnection logic here, so a dropped terminal connection just stops the output.

In [ ]:
streaming_straddle()

This is a minimal demonstration, with no check that the two cached quotes are recent and no reconnection if the terminal drops. Any listed contract we hold entitlements for works the same way, though a four-leg structure needs more cache slots and a rule for when enough legs have quoted. The mark we calculate is still ours rather than an executable price, but each leg behind it is a live exchange quote instead of a 15-minute-old snapshot.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.